# EDA — como os dados do TSE chegam

Objetivo: entender **na prática** o formato de divulgação do TSE (JSON no CDN),
antes de modelar cache, API e dashboard.

Fonte oficial:
- Base: `https://resultados.tse.jus.br`
- Ambiente: `oficial`
- Ciclo: `ele2026`
- 1º turno federal: eleição `6257` (pleito `3220`)

> Não é uma API REST com query params. É **arquivo JSON** com URL previsível.

## 0. Setup

Rode o notebook com o kernel do `.venv` do projeto.

No terminal (se ainda não instalou):
```bash
source .venv/bin/activate
pip install jupyter ipykernel pandas
python -m ipykernel install --user --name tse-eleicao --display-name "Python (TSE)"
```

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from pprint import pprint
from typing import Any

import httpx
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

BASE = "https://resultados.tse.jus.br"
AMBIENTE = "oficial"
CICLO = "ele2026"
ELEICAO_FEDERAL_1T = "6257"
TIMEOUT = 30.0

client = httpx.Client(timeout=TIMEOUT, follow_redirects=True)
print("ok — client httpx pronto")

## 1. Mapa de arquivos (config global)

O ponto de entrada clássico é o **EA11** / config de eleições:
`/oficial/comum/config/ele-c.json`

Ele lista pleitos, códigos de eleição e **templates de diretório** (`arq`).

In [ ]:
url_config = f"{BASE}/{AMBIENTE}/comum/config/ele-c.json"
resp_cfg = client.get(url_config)
print("HTTP", resp_cfg.status_code, "|", url_config)
print("content-type:", resp_cfg.headers.get("content-type"))
print("cache-control:", resp_cfg.headers.get("cache-control"))
print("cdn-cache-status:", resp_cfg.headers.get("cdn-cache-status"))
print("x-ratelimit-limit:", resp_cfg.headers.get("x-ratelimit-limit"))
print("x-ratelimit-remaining:", resp_cfg.headers.get("x-ratelimit-remaining"))

cfg = resp_cfg.json()
print("\nchaves top-level:", list(cfg.keys()))
print("gerado (dg/hg):", cfg.get("dg"), cfg.get("hg"))

In [ ]:
# Templates de pasta — é daqui que sai o padrão de URL
print("=== templates em cfg['arq'] ===")
for item in cfg.get("arq", []):
    print(f"  tp={item.get('tp'):<4}  dir={item.get('dir')}")

In [ ]:
# Pleitos / eleições — em 2026: pl[].e[] (ciclo fica em pl[].c)
rows = []
for pleito in cfg.get("pl", []):
    for ele in pleito.get("e", []) or []:
        rows.append(
            {
                "ciclo": pleito.get("c"),
                "pleito": pleito.get("cd"),
                "data_pleito": pleito.get("dt"),
                "cd_eleicao": ele.get("cd"),
                "nome": ele.get("nm"),
                "turno": ele.get("t"),
                "tipo": ele.get("tp"),
                "cd_2turno": ele.get("cdt2"),
            }
        )

df_eleicoes = pd.DataFrame(rows)
mask_ciclo = df_eleicoes["ciclo"].eq("ele2026")
display(
    df_eleicoes.loc[mask_ciclo]
    .sort_values(["pleito", "cd_eleicao"])
    .reset_index(drop=True)
)
print(f"eleições ele2026: {mask_ciclo.sum()} | total no config: {len(df_eleicoes)}")

## 2. Baixar o resultado unificado (Presidente BR — 1º turno)

Padrão de nome (EA20):

`{abr}-c{cargo_4dig}-e{eleicao_6dig}-u.json`

Exemplo:
`br-c0001-e006257-u.json`

In [ ]:
def url_resultado(cd_eleicao: str, abr: str = "br", cargo: str = "1") -> str:
    abr = abr.lower()
    nome = f"{abr}-c{cargo.zfill(4)}-e{cd_eleicao.zfill(6)}-u.json"
    return f"{BASE}/{AMBIENTE}/{CICLO}/{cd_eleicao}/dados/{abr}/{nome}"


url_br = url_resultado(ELEICAO_FEDERAL_1T, "br", "1")
print(url_br)

resp = client.get(url_br)
print("HTTP", resp.status_code)
print("bytes:", len(resp.content))
print("cache-control:", resp.headers.get("cache-control"))
print("cdn-cache-status:", resp.headers.get("cdn-cache-status"))
print("etag / last-modified:", resp.headers.get("etag"), resp.headers.get("last-modified"))

data = resp.json()
print("tipo:", type(data).__name__, "| chaves:", list(data.keys()))

## 3. Camada 1 — metadados do arquivo

Campos úteis no root (nomes curtos = padrão TSE):

| campo | significado típico |
|-------|--------------------|
| `ele` | código da eleição |
| `t` | turno |
| `tpabr` / `cdabr` | tipo e código da abrangência (`br`, `sp`...) |
| `dg` / `hg` | data/hora de **geração** do arquivo |
| `dt` / `ht` | data/hora de **totalização** |
| `tf` | totalização final? (`s`/`n`) |
| `and` | andamento (`f` = finalizado, etc.) |
| `s` | bloco de **seções** |
| `e` | bloco de **eleitorado** |
| `v` | bloco de **votos** agregados |
| `carg` | cargos + candidatos |

In [ ]:
meta_keys = [k for k in data.keys() if k not in {"s", "e", "v", "carg", "mnae"}]
meta = {k: data[k] for k in meta_keys}
pprint(meta, sort_dicts=False)

## 4. Bloco `s` — seções eleitorais

`pst` / `pstn` costumam ser o % de seções totalizadas (string com vírgula vs número).

In [ ]:
s = data.get("s") or {}
print("chaves em s:", list(s.keys()))
df_s = pd.DataFrame([{"campo": k, "valor": v} for k, v in s.items()])
display(df_s)

print("\nLeitura rápida:")
print(f"  seções totais (ts):        {s.get('ts')}")
print(f"  seções totalizadas (st):   {s.get('st')}  → {s.get('pst')}%")
print(f"  seções não totaliz. (snt): {s.get('snt')}  → {s.get('psnt')}%")

## 5. Bloco `e` — eleitorado / comparecimento

In [ ]:
e = data.get("e") or {}
df_e = pd.DataFrame([{"campo": k, "valor": v} for k, v in e.items()])
display(df_e)

print("\nLeitura rápida:")
print(f"  eleitorado (te):     {e.get('te')}")
print(f"  comparecimento (c):  {e.get('c')}  → {e.get('pc')}%")
print(f"  abstenção (a):       {e.get('a')}  → {e.get('pa')}%")

## 6. Bloco `v` — votos agregados (brancos, nulos, válidos...)

In [ ]:
v = data.get("v") or {}
df_v = pd.DataFrame([{"campo": k, "valor": v_} for k, v_ in v.items()])
display(df_v)

print("\nLeitura rápida:")
print(f"  total de votos (tv):     {v.get('tv')}")
print(f"  votos válidos (vv/vvc):  {v.get('vv')} / {v.get('vvc')}")
print(f"  brancos (vb):            {v.get('vb')}  → {v.get('pvb')}%")
print(f"  nulos (tvn):             {v.get('tvn')} → {v.get('ptvn')}%")

## 7. Bloco `carg` — cargos, federações e candidatos

Hierarquia observada no 1º turno:

```
carg[]
  └─ agr[]          # agrupamentos
       └─ par[]     # partido / coligação
            └─ cand[]  # candidatos (n, nm, vap, pvap, ...)
  └─ fed[]          # federações (metadado)
```

In [ ]:
carg_list = data.get("carg") or []
print(f"qtd cargos no arquivo: {len(carg_list)}")

for cargo in carg_list:
    print(
        f"- cd={cargo.get('cd')}  nmn={cargo.get('nmn')}  "
        f"agr={len(cargo.get('agr') or [])}  fed={len(cargo.get('fed') or [])}"
    )

cargo = carg_list[0]
print("\nchaves do 1º cargo:", list(cargo.keys()))
print("exemplo agr[0] keys:", list((cargo.get("agr") or [{}])[0].keys()))
print("exemplo par[0] keys:", list(((cargo.get("agr") or [{}])[0].get("par") or [{}])[0].keys()))
cand0 = (((cargo.get("agr") or [{}])[0].get("par") or [{}])[0].get("cand") or [{}])[0]
print("exemplo cand[0]:")
pprint(cand0, sort_dicts=False)

In [ ]:
def candidatos_para_df(payload: dict[str, Any]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for cargo in payload.get("carg") or []:
        for agr in cargo.get("agr") or []:
            for par in agr.get("par") or []:
                for cand in par.get("cand") or []:
                    rows.append(
                        {
                            "cargo_cd": cargo.get("cd"),
                            "cargo": cargo.get("nmn"),
                            "agr_n": agr.get("n"),
                            "agr_nm": agr.get("nm"),
                            "partido_n": par.get("n"),
                            "partido_sg": par.get("sg"),
                            "partido_nm": par.get("nm"),
                            "numero": cand.get("n"),
                            "nome": cand.get("nm"),
                            "votos": int(str(cand.get("vap") or "0").replace(".", "")),
                            "pct": cand.get("pvap"),
                            "situacao": cand.get("st") or cand.get("dvt") or cand.get("seq"),
                            "raw_keys": ",".join(sorted(cand.keys())),
                        }
                    )
    return pd.DataFrame(rows).sort_values("votos", ascending=False).reset_index(drop=True)


df_cand = candidatos_para_df(data)
print("shape:", df_cand.shape)
print("chaves que aparecem nos candidatos:")
print(sorted({k for ks in df_cand["raw_keys"] for k in ks.split(",") if k}))
display(df_cand.drop(columns=["raw_keys"]))

## 8. Mesmo cargo, outra abrangência (UF)

O arquivo `br` é nacional. Também existem arquivos por UF
(`sp-c0001-e006257-u.json`, etc.). Vale comparar tamanho e se o schema é igual.

In [ ]:
ufs = ["sp", "rj", "mg", "ba"]
resumo_ufs = []

for uf in ufs:
    url = url_resultado(ELEICAO_FEDERAL_1T, uf, "1")
    r = client.get(url)
    if r.status_code != 200:
        resumo_ufs.append({"uf": uf, "http": r.status_code, "bytes": None, "pst": None, "top": None})
        continue
    payload = r.json()
    df_uf = candidatos_para_df(payload)
    top = df_uf.iloc[0]["nome"] if len(df_uf) else None
    resumo_ufs.append(
        {
            "uf": uf,
            "http": r.status_code,
            "bytes": len(r.content),
            "pst": (payload.get("s") or {}).get("pst"),
            "chaves": list(payload.keys()),
            "top": top,
            "votos_top": int(df_uf.iloc[0]["votos"]) if len(df_uf) else None,
        }
    )

df_ufs = pd.DataFrame(resumo_ufs)
display(df_ufs.drop(columns=["chaves"], errors="ignore"))

# schema idêntico ao BR?
if df_ufs["http"].eq(200).any():
    amostra = next(x for x in resumo_ufs if x["http"] == 200)
    print("chaves UF == chaves BR?", amostra["chaves"] == list(data.keys()))
    print("chaves UF:", amostra["chaves"])

## 9. Implicações pra arquitetura (anotações de estudo)

Depois de olhar o payload, algumas decisões ficam óbvias:

1. **URL determinística** → fácil cachear por chave (`eleicao:abr:cargo`).
2. **Arquivo grande e aninhado** → o BFF deve devolver um **resumo** pro front, não o JSON bruto.
3. Headers (`cache-control`, `cdn-cache-status`, rate limit) → o TSE já tem CDN;
   nosso cache evita *nossa* infra e o rate limit (100 req/IP/s oficialmente).
4. Campos `dg`/`hg` / `idg` → dá pra detectar "arquivo mudou" sem reprocessar tudo.
5. Números vêm como **string** e % com **vírgula** → normalizar na borda do cliente.

Próximos notebooks possíveis:
- parse estadual (governador `6259`)
- evolução temporal (poll + diff de `idg`)
- desenho do schema do nosso cache/API

In [ ]:
# Salva uma cópia local do JSON BR pra estudar offline (opcional)
out_dir = Path("../data/raw")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"br-c0001-e{ELEICAO_FEDERAL_1T.zfill(6)}-u.json"
out_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
print("salvo em", out_path.resolve())
print("tamanho MB:", round(out_path.stat().st_size / 1_048_576, 3))

client.close()
print("client fechado")